# Extract PDF Tables & Fields with Azure AI Document Intelligence

Extracts **only the tabular data** from a PDF using **Azure AI Document Intelligence**
and writes it to local Parquet/CSV. It is **PDF-agnostic**: point it at any PDF and it
returns whatever tables the service detects — nothing about the document's layout,
wording, or schedule names is hard-coded. The AT&T 2Q2025 Financial & Operational
Schedules PDF is just the running example.

This is the cloud-AI counterpart to `PDF-to-Table.ipynb` (open-source `pdfplumber`).
It runs **two kinds of model** in a single pass:

| Model | What it returns | Used for |
|---|---|---|
| `prebuilt-layout` | `tables` (cells, spans, header roles) + `pages` | Structured tables, no training |
| **Custom extraction** (your trained model) | `documents[].fields` (named key/values) | Pulling specific fields you defined |

Set `CUSTOM_MODEL_ID` to your trained model's id to enable field extraction; leave it
`None` for tables-only. Each model's response is cached locally (per file + model) so
re-running does not re-call — and re-bill — the service.

**Pipeline:** configure → check deps → locate PDF → authenticate → analyse each
model (cached) → tables → DataFrames → fields → DataFrame → normalise → write.

### Robust to PDF changes
- **Detection, not hard-coding.** Tables come from Azure's `prebuilt-layout`, so a new
  layout, renamed schedules, or reordered columns still parse correctly.
- **Tabular-data quality gate.** `MIN_TABLE_ROWS` / `MIN_TABLE_COLS` / `MIN_TABLE_FILL`
  drop stray, near-empty layout boxes so you keep only genuine tables. Loosen them if a
  real table is being skipped; tighten them if noise slips through.
- **Auto-named outputs.** With `OUTPUT_PREFIX = None`, output files are named after the
  PDF, so a different PDF never overwrites a previous run.
- **Just drop the file in.** With `PDF_FILENAME = None`, the most recently modified PDF
  in `RawData/PDF` is used; a changed file (new size) automatically triggers a fresh
  analysis because the cache is keyed by file name + size.

### Resource & credentials
- Resource: **Azure AI Document Intelligence**, pricing tier **S0 (Standard)**.
- Provide credentials in **either** of two ways (the config cell checks inline first,
  then environment variables):

  **A. Inline (quickest)** — paste into the *Credentials* section of the config cell:

  ```python
  AZURE_ENDPOINT = "https://<your-resource>.cognitiveservices.azure.com/"
  AZURE_KEY = "<your-key>"
  ```

  ⚠️ **Do not commit a real key.** If this notebook is tracked in git, leave
  `AZURE_KEY = ""` and supply the key via the environment variable instead (option B).

  **B. Environment variables (safer for secrets)** — leave the inline values blank and set:

  ```powershell
  setx AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT "https://<your-resource>.cognitiveservices.azure.com/"
  setx AZURE_DOCUMENT_INTELLIGENCE_KEY "<your-key>"
  ```

  (Restart VS Code after `setx` so the kernel inherits the new variables.)

  **C. Keyless** — leave the key blank and unset to use `DefaultAzureCredential`
  (`az login` / managed identity). The principal needs the **Cognitive Services User**
  role on the resource.


In [ ]:
# --- Configuration -----------------------------------------------------------
import os
from pathlib import Path

# Optional: pin a specific file in RawData/PDF. If None, the most recently modified
# *.pdf is used (so dropping in a new PDF "just works").
PDF_FILENAME: str | None = None

# Models to run. prebuilt-layout gives tables; set CUSTOM_MODEL_ID to your trained
# custom extraction model id to also pull named fields. Leave it None for layout only.
LAYOUT_MODEL_ID = "prebuilt-layout"
CUSTOM_MODEL_ID: str | None = None  # e.g. "att-schedules-extractor" (your model id)
MODELS = [m for m in (LAYOUT_MODEL_ID, CUSTOM_MODEL_ID) if m]

# --- Credentials -------------------------------------------------------------
# Paste your values below, OR leave them blank to read from environment variables.
# The endpoint is not a secret. Do NOT paste a real KEY here if this notebook is
# committed to git -- keep the key in the AZURE_DOCUMENT_INTELLIGENCE_KEY env var.
AZURE_ENDPOINT = "https://vtfabricpoc.cognitiveservices.azure.com/"
AZURE_KEY = ""  # blank -> use the env var below, or DefaultAzureCredential (az login)

# Fallback environment-variable names (used only when the inline values are blank).
ENDPOINT_ENV = "AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"
KEY_ENV = "AZURE_DOCUMENT_INTELLIGENCE_KEY"

# "Only tabular data" quality gate. Azure sometimes returns tiny or near-empty
# layout boxes as tables; keep only those that look like genuine tables.
MIN_TABLE_ROWS = 2     # at least this many rows
MIN_TABLE_COLS = 2     # at least this many columns
MIN_TABLE_FILL = 0.20  # at least this fraction of cells must be non-empty

# Output file-name prefix. None -> auto-derived from the PDF file name, so a
# different PDF produces differently-named outputs (no accidental overwrite).
OUTPUT_PREFIX: str | None = None
# Re-analysis is cached on disk (per file + model); True bypasses it and re-calls.
FORCE_REANALYZE = False


def find_project_root(start: Path) -> Path:
    """Walk up from `start` until a folder containing 'RawData' is found."""
    for candidate in (start, *start.parents):
        if (candidate / "RawData").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root (a parent of {start} containing 'RawData')."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
PDF_DIR = PROJECT_ROOT / "RawData" / "PDF"
OUTPUT_DIR = PROJECT_ROOT / "RawData" / "scratch" / "extracted_pdf_tables_azure"
CACHE_DIR = OUTPUT_DIR / "_cache"

print(f"Project root : {PROJECT_ROOT}")
print(f"PDF dir      : {PDF_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Models       : {MODELS}")

In [ ]:
# --- Dependencies ------------------------------------------------------------
# Fail fast with an actionable message instead of installing packages at runtime.
import importlib.util
import sys

_REQUIRED = {
    "azure.ai.documentintelligence": "azure-ai-documentintelligence",
    "azure.identity": "azure-identity",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
}
_missing = [pip_name for mod, pip_name in _REQUIRED.items()
            if importlib.util.find_spec(mod) is None]
if _missing:
    raise ModuleNotFoundError(
        "Missing required packages: " + ", ".join(_missing)
        + f"\nInstall them with:\n    {sys.executable} -m pip install "
        + " ".join(_missing)
    )

import json
import re

import pandas as pd
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError

import azure.ai.documentintelligence as _di

print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | "
      f"azure-ai-documentintelligence {_di.VERSION}")


In [ ]:
# --- Locate and validate the PDF --------------------------------------------
def resolve_pdf(pdf_dir: Path, filename: str | None) -> Path:
    """Return the PDF to process, validating it exists and is non-empty.

    With no `filename`, the most recently modified *.pdf is used so that dropping
    a new file into the folder is picked up automatically.
    """
    if not pdf_dir.is_dir():
        raise FileNotFoundError(f"PDF directory does not exist: {pdf_dir}")

    if filename:
        path = pdf_dir / filename
        if not path.is_file():
            raise FileNotFoundError(f"Configured PDF not found: {path}")
    else:
        pdfs = sorted(pdf_dir.glob("*.pdf"))
        if not pdfs:
            raise FileNotFoundError(f"No '*.pdf' files found in {pdf_dir}")
        if len(pdfs) > 1:
            pdfs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
            print(f"Multiple PDFs found; using the most recently modified: "
                  f"{pdfs[0].name}")
        path = pdfs[0]

    if path.stat().st_size == 0:
        raise ValueError(f"PDF is empty: {path}")
    return path


PDF_PATH = resolve_pdf(PDF_DIR, PDF_FILENAME)
print(f"Using PDF: {PDF_PATH.name}  ({PDF_PATH.stat().st_size / 1_048_576:.2f} MB)")

In [ ]:
# --- Authenticate and build the client --------------------------------------
# Endpoint and key come from the inline values in the config cell, falling back to
# environment variables when those are blank. If no key is available we use
# DefaultAzureCredential (az login / managed identity / VS Code sign-in).
def build_client() -> DocumentIntelligenceClient:
    endpoint = (AZURE_ENDPOINT or os.environ.get(ENDPOINT_ENV, "")).strip()
    if not endpoint:
        raise EnvironmentError(
            "No endpoint configured. Set AZURE_ENDPOINT in the config cell, or the "
            f"{ENDPOINT_ENV} environment variable, e.g. "
            "https://<resource>.cognitiveservices.azure.com/"
        )

    key = (AZURE_KEY or os.environ.get(KEY_ENV, "")).strip()
    if key:
        credential = AzureKeyCredential(key)
        auth = "API key"
    else:
        # Imported lazily so the notebook still runs key-based without azure-identity.
        from azure.identity import DefaultAzureCredential
        credential = DefaultAzureCredential()
        auth = "DefaultAzureCredential (Azure AD)"

    print(f"Endpoint : {endpoint}")
    print(f"Auth     : {auth}")
    return DocumentIntelligenceClient(endpoint=endpoint, credential=credential)


client = build_client()

In [ ]:
# --- Analyse the document with each model (on-disk caching, per model) -------
# Results are cached as JSON keyed by file name + size + model id, so re-running
# the notebook does not re-call (and re-bill) the service. Delete the cache files
# or set FORCE_REANALYZE = True to force fresh analyses.
def analyze(client: DocumentIntelligenceClient, pdf_path: Path, model_id: str) -> dict:
    """Return one model's result as a plain dict (wire/JSON shape)."""
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    safe_model = re.sub(r"[^A-Za-z0-9._-]+", "_", model_id)
    cache_file = CACHE_DIR / f"{pdf_path.stem}_{safe_model}_{pdf_path.stat().st_size}.json"

    if cache_file.is_file() and not FORCE_REANALYZE:
        print(f"[{model_id}] loading cached result: {cache_file.name}")
        return json.loads(cache_file.read_text(encoding="utf-8"))

    print(f"[{model_id}] analysing '{pdf_path.name}' ... (this calls Azure)")
    data = pdf_path.read_bytes()
    try:
        poller = client.begin_analyze_document(
            model_id, AnalyzeDocumentRequest(bytes_source=data)
        )
        result = poller.result()
    except HttpResponseError as exc:
        raise RuntimeError(
            f"Azure analysis failed for model '{model_id}': {exc.status_code} "
            f"{exc.reason}. Check the endpoint, key/role, that the model id exists, "
            "and that the resource is active."
        ) from exc

    result_dict = result.as_dict()
    cache_file.write_text(json.dumps(result_dict), encoding="utf-8")
    print(f"[{model_id}] done -> cached to {cache_file.name}")
    return result_dict


results = {model_id: analyze(client, PDF_PATH, model_id) for model_id in MODELS}

for model_id, layout in results.items():
    print(f"\n{model_id}: "
          f"{len(layout.get('pages', []))} pages, "
          f"{len(layout.get('tables', []))} tables, "
          f"{len(layout.get('documents', []))} documents")


In [ ]:
# --- Convert Azure tables to DataFrames -------------------------------------
# Azure returns each table as a flat list of cells with row/column indices, spans
# and a 'kind' (e.g. 'columnHeader'). We rebuild a 2-D grid per table, expand
# merged cells across their span, and promote leading header rows to column names.
# Nothing here is specific to a particular PDF: every table the service detects is
# processed, and a quality gate keeps only genuine tables.
def _g(d: dict, *names: str, default=None):
    """Get the first present key from camelCase/snake_case variants."""
    for n in names:
        if n in d:
            return d[n]
    return default


def _gi(d: dict, *names: str, default: int = 0) -> int:
    """Like _g, but always returns an int (Azure cell indices/counts/spans)."""
    value = _g(d, *names, default=default)
    return int(value) if value is not None else default


def _table_page(table: dict) -> int | None:
    regions = _g(table, "boundingRegions", "bounding_regions", default=[]) or []
    if regions:
        return _g(regions[0], "pageNumber", "page_number")
    return None


def _table_caption(table: dict) -> str:
    """The table's caption text, if Azure detected one (a generic table title)."""
    caption = _g(table, "caption", default=None)
    if isinstance(caption, dict):
        return (_g(caption, "content", default="") or "").replace("\n", " ").strip()
    return ""


def table_to_grid(table: dict) -> list[list[str]]:
    """Build a dense 2-D string grid, expanding row/column spans."""
    rows = _gi(table, "rowCount", "row_count")
    cols = _gi(table, "columnCount", "column_count")
    grid = [["" for _ in range(cols)] for _ in range(rows)]
    for cell in _g(table, "cells", default=[]) or []:
        r = _gi(cell, "rowIndex", "row_index")
        c = _gi(cell, "columnIndex", "column_index")
        rs = _gi(cell, "rowSpan", "row_span", default=1) or 1
        cs = _gi(cell, "columnSpan", "column_span", default=1) or 1
        text = (_g(cell, "content", default="") or "").replace("\n", " ").strip()
        for dr in range(rs):
            for dc in range(cs):
                rr, cc = r + dr, c + dc
                if rr < rows and cc < cols:
                    grid[rr][cc] = text
    return grid


def is_tabular(grid: list[list[str]]) -> bool:
    """Quality gate: keep only grids that look like genuine tables, not stray
    single-cell/near-empty layout boxes. Tuned via MIN_TABLE_* in the config cell."""
    if len(grid) < MIN_TABLE_ROWS:
        return False
    n_cols = len(grid[0]) if grid else 0
    if n_cols < MIN_TABLE_COLS:
        return False
    total = sum(len(row) for row in grid)
    filled = sum(1 for row in grid for value in row if value.strip())
    return total > 0 and (filled / total) >= MIN_TABLE_FILL


def _header_depth(table: dict, total_rows: int) -> int:
    """Count leading rows whose cells are all column headers."""
    header_rows = {
        _gi(cell, "rowIndex", "row_index")
        for cell in _g(table, "cells", default=[]) or []
        if _g(cell, "kind") == "columnHeader"
    }
    depth = 0
    while depth in header_rows and depth < total_rows:
        depth += 1
    return depth


def grid_to_dataframe(grid: list[list[str]], depth: int) -> pd.DataFrame:
    """Turn a grid into a DataFrame, promoting `depth` leading rows to headers."""
    if depth == 0:
        df = pd.DataFrame(grid)
        df.columns = [f"col_{i + 1}" for i in range(df.shape[1])]
        return df

    # Join multi-row headers with ' / '; de-duplicate blanks into positional names.
    header_rows = grid[:depth]
    names: list[str] = []
    for c in range(len(grid[0])):
        parts = [header_rows[r][c] for r in range(depth) if header_rows[r][c]]
        names.append(" / ".join(dict.fromkeys(parts)) or f"col_{c + 1}")
    seen: dict[str, int] = {}
    unique: list[str] = []
    for name in names:
        seen[name] = seen.get(name, 0) + 1
        unique.append(name if seen[name] == 1 else f"{name}_{seen[name]}")
    return pd.DataFrame(grid[depth:], columns=unique)


# (model_id, table_index, page, caption, df) for every table that passes the gate.
table_frames: list[tuple[str, int, int, str, pd.DataFrame]] = []
skipped = 0
for model_id, layout in results.items():
    for idx, table in enumerate(layout.get("tables", [])):
        grid = table_to_grid(table)
        if not is_tabular(grid):
            skipped += 1
            continue
        df = grid_to_dataframe(grid, _header_depth(table, len(grid)))
        table_frames.append(
            (model_id, idx, _table_page(table) or 0, _table_caption(table), df)
        )

print(f"Kept {len(table_frames)} tables across {len(results)} model(s); "
      f"skipped {skipped} that failed the tabular-data gate.")
if table_frames:
    _m, _i, _p, _cap, _df = table_frames[0]
    print(f"\nExample - {_m} table {_i} (page {_p})"
          f"{f' — {_cap}' if _cap else ''}, shape {_df.shape}:")
    display(_df.head(10))

In [ ]:
# --- Combine tables into one tidy long-format frame -------------------------
# Each table keeps its own columns, so we stack them with model/table/page/caption
# keys and a positional column id.
combined_rows: list[dict] = []
for model_id, table_index, page, caption, df in table_frames:
    headers = list(df.columns)
    for _, record in df.iterrows():
        row = {
            "model_id": model_id,
            "table_index": table_index,
            "page": page,
            "caption": caption,
        }
        for pos, header in enumerate(headers, start=1):
            row[f"col_{pos}"] = record[header]
        combined_rows.append(row)

combined_df = pd.DataFrame(combined_rows)
if combined_df.empty:
    print("No tables were extracted. Either the PDF has no tables, or they were "
          "filtered out by the quality gate (lower MIN_TABLE_* in the config cell).")
else:
    print(f"Combined long frame: {combined_df.shape[0]} rows, {combined_df.shape[1]} cols")
combined_df.head(20)

In [ ]:
# --- Extract custom-model fields (key/value) --------------------------------
# Custom extraction models return `documents`, each with named `fields`. Prebuilt
# layout returns none, so this frame is empty unless a custom model ran.
_VALUE_KEYS = (
    "valueString", "valueNumber", "valueInteger", "valueDate", "valueTime",
    "valuePhoneNumber", "valueCountryRegion", "valueBoolean", "valueSelectionMark",
)


def field_value(field: dict):
    """Best scalar value for a field, falling back to its raw 'content'."""
    for key in _VALUE_KEYS:
        if field.get(key) is not None:
            return field[key]
    return field.get("content")  # arrays/objects keep their raw text


def extract_fields(results: dict) -> pd.DataFrame:
    """Flatten documents[].fields from every model into one tidy frame."""
    rows: list[dict] = []
    for model_id, layout in results.items():
        for doc_index, document in enumerate(layout.get("documents", [])):
            for name, field in (document.get("fields") or {}).items():
                field = field or {}
                rows.append({
                    "model_id": model_id,
                    "doc_index": doc_index,
                    "doc_type": document.get("docType") or document.get("doc_type"),
                    "field": name,
                    "value": field_value(field),
                    "type": field.get("type"),
                    "confidence": field.get("confidence"),
                })
    return pd.DataFrame(rows)


fields_df = extract_fields(results)
if fields_df.empty:
    print("No custom-model fields found (set CUSTOM_MODEL_ID to enable field extraction).")
else:
    print(f"Extracted {len(fields_df)} fields from custom model(s).")
fields_df.head(30)


In [ ]:
# --- Normalise money/percent values to numeric ------------------------------
def to_number(value) -> float | None:
    """Convert a financial string ('$ 1,234', '(56)', '7.9 %') to a float.

    Parentheses denote negatives; nil/blank/dash values become None (which
    pandas treats as NA when the column is cast to Float64).
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if s in ("", "-", "—", "�"):  # blank, hyphen, em-dash, replacement char
        return None
    negative = s.startswith("(") and s.endswith(")")
    s = re.sub(r"[\$,%()\s—]", "", s)
    if s in ("", "-"):
        return None
    try:
        number = float(s)
    except ValueError:
        return None
    return -number if negative else number


def numeric_view(df: pd.DataFrame) -> pd.DataFrame:
    """Copy with every mostly-numeric column cast to Float64 (keys untouched)."""
    out = df.copy()
    key_cols = {"model_id", "table_index", "page", "caption"}
    for col in out.columns:
        if col in key_cols:
            continue
        converted = out[col].map(to_number)
        non_null = converted.notna().sum()
        present = out[col].astype("string").str.strip().replace("", pd.NA).notna().sum()
        # Only treat as a value column if most populated cells parse as numbers.
        if present and non_null >= max(1, int(0.6 * present)):
            out[col] = converted.astype("Float64")
    return out


numeric_df = numeric_view(combined_df) if not combined_df.empty else combined_df
print("Normalised numeric columns where the values are mostly numbers.")
numeric_df.head(20)

In [ ]:
# --- Write outputs -----------------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _slug(text: str, max_len: int = 40) -> str:
    """Lowercase, filesystem-safe slug from arbitrary text."""
    return re.sub(r"[^A-Za-z0-9]+", "_", text or "").strip("_").lower()[:max_len]


# Prefix auto-derives from the PDF name unless OUTPUT_PREFIX is set, so a new PDF
# writes to new files instead of overwriting the previous run.
prefix = OUTPUT_PREFIX or f"{_slug(PDF_PATH.stem, 60)}_"
written: list[Path] = []

if not combined_df.empty:
    combined_parquet = OUTPUT_DIR / f"{prefix}tables.parquet"
    combined_csv = OUTPUT_DIR / f"{prefix}tables.csv"
    numeric_parquet = OUTPUT_DIR / f"{prefix}tables_numeric.parquet"
    combined_df.to_parquet(combined_parquet, index=False)
    combined_df.to_csv(combined_csv, index=False, encoding="utf-8-sig")
    numeric_df.to_parquet(numeric_parquet, index=False)
    written += [combined_parquet, combined_csv, numeric_parquet]

    # One CSV per detected table, with its native headers preserved.
    by_table_dir = OUTPUT_DIR / f"{prefix}by_table"
    by_table_dir.mkdir(exist_ok=True)
    for model_id, table_index, page, caption, df in table_frames:
        safe_model = re.sub(r"[^A-Za-z0-9._-]+", "_", model_id)
        name = f"{safe_model}_t{table_index:02d}_p{page:02d}"
        if caption:
            name += f"_{_slug(caption)}"
        df.to_csv(by_table_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"Wrote {len(table_frames)} per-table CSVs in {by_table_dir}")

if not fields_df.empty:
    fields_csv = OUTPUT_DIR / f"{prefix}fields.csv"
    fields_parquet = OUTPUT_DIR / f"{prefix}fields.parquet"
    fields_df.to_csv(fields_csv, index=False, encoding="utf-8-sig")
    fields_df.to_parquet(fields_parquet, index=False)
    written += [fields_csv, fields_parquet]

if written:
    print("Wrote:")
    for path in written:
        print(f"  {path}  ({path.stat().st_size:,} bytes)")
else:
    print("Nothing to write — no tables or fields were extracted.")